## Silver — `cno` (Cadastro Nacional de Obras)

**Origem:** `workspace.bronze.cno` → **Destino:** `workspace.silver.cno`

- **Grão:** 1 linha por obra — em caso de duplicidade de `cno`, mantém o registro com `data_da_situacao` mais recente.
- **Transformações:**
  - Seleção apenas das colunas relevantes à análise: `cno`, `data_de_inicio`, `codigo_do_municipio`, `unidade_de_medida`, `area_total`, `situacao`, `data_da_situacao`.
  - Correção de tipos: datas convertidas para `date` (`yyyy-MM-dd`) e `area_total` para `double`; registros que não convertem viram nulos e são descartados.
  - Filtros de qualidade (cada regra contabilizada no relatório abaixo):
    - `cno` não nulo/vazio;
    - `data_de_inicio` e `data_da_situacao` válidas;
    - `codigo_do_municipio` não nulo, numérico e **com registro em** `workspace.silver.municipios` (código IBGE de 7 dígitos);
    - `unidade_de_medida` não nula;
    - `area_total` não nula e >= 0;
    - `situacao` dentro do domínio oficial RFB (01, 02, 03, 14, 15).
- **Relatório de qualidade:** quantidade de inválidos por regra + % sobre o total bronze.
- **Linhagem:** CSV dados.gov.br → `bronze.cno` → limpeza/validação → `silver.cno`.

In [0]:
%run ./_setup_cno

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from data_pipeline import save_table, add_column_comments, resumo_invalidos, condicao_valida
from metadata.metadata import SILVER_CNO_COMMENTS, DOMINIO_SITUACAO

In [0]:
SOURCE_TABLE = "workspace.bronze.cno"
TARGET_TABLE = "workspace.silver.cno"
MUNICIPIOS_TABLE = "workspace.silver.municipios"

# Contrato de saída silver.cno
COLUNAS_FINAIS = [
    "cno",
    "data_de_inicio",
    "codigo_do_municipio",
    "unidade_de_medida",
    "area_total",
    "situacao",
    "data_da_situacao",
]

In [0]:
df_bronze = spark.table(SOURCE_TABLE)
total_bronze = df_bronze.count()
print(f"Bronze: {total_bronze:,} linhas | colunas: {df_bronze.columns}")

# Seleção das colunas relevantes com cast seguro:
# datas -> date (yyyy-MM-dd); area_total -> double; códigos/texto -> string trim
df = df_bronze.select(
    F.trim(F.col("cno").cast("string")).alias("cno"),
    F.to_date(F.col("data_de_inicio").cast("string"), "yyyy-MM-dd").alias("data_de_inicio"),
    F.trim(F.col("codigo_do_municipio").cast("string")).alias("codigo_do_municipio"),
    F.trim(F.col("unidade_de_medida").cast("string")).alias("unidade_de_medida"),
    F.col("area_total").cast("string").cast("double").alias("area_total"),
    F.trim(F.col("situacao").cast("string")).alias("situacao"),
    F.to_date(F.col("data_da_situacao").cast("string"), "yyyy-MM-dd").alias("data_da_situacao"),
)
display(df.limit(5))

In [0]:
# Left join com a base de municípios: sem correspondência => código inválido
df_municipios = (
    spark.table(MUNICIPIOS_TABLE)
    .select("codigo_municipio")
    .distinct()
    .withColumn("municipio_valido", F.lit(True))
)

df = df.join(df_municipios, on="codigo_municipio", how="left")
com_registro = df.filter(F.col("municipio_valido")).count()
pct_registro = round(100 * com_registro / total_bronze, 2) if total_bronze else 0.0
print(f"Códigos com registro em municipios: {com_registro:,} ({pct_registro}%)")
if pct_registro < 90:
    print("ALERTA: baixa adesão ao join — verificar se o CNO usa TOM em vez de IBGE ou formatos divergentes.")

In [0]:
REGRAS_INVALIDOS = {
    "cno_nulo_ou_vazio": F.col("cno").isNull() | (F.col("cno") == ""),
    "data_de_inicio_nula_ou_invalida": F.col("data_de_inicio").isNull(),
    "codigo_do_municipio_nulo_ou_nao_numerico": (
        F.col("codigo_do_municipio").isNull()
        | ~F.col("codigo_do_municipio").rlike("^[0-9]{1,7}$")
    ),
    "codigo_do_municipio_sem_registro_em_municipios": ~F.coalesce(F.col("municipio_valido"), F.lit(False)),
    "unidade_de_medida_nula": F.col("unidade_de_medida").isNull() | (F.col("unidade_de_medida") == ""),
    "area_total_nula_ou_negativa": F.col("area_total").isNull() | (F.col("area_total") < 0),
    "situacao_fora_do_dominio": ~F.coalesce(F.col("situacao").isin(list(DOMINIO_SITUACAO)), F.lit(False)),
    "data_da_situacao_nula_ou_invalida": F.col("data_da_situacao").isNull(),
}

In [0]:
df_relatorio = resumo_invalidos(spark, df, REGRAS_INVALIDOS)
print(f"Total bronze avaliado: {total_bronze:,}")
display(df_relatorio)

In [0]:
# Mantém apenas registros válidos em todas as regras
df = df.filter(condicao_valida(REGRAS_INVALIDOS))

# Deduplicação determinística: mantém a situação mais recente da obra
w = Window.partitionBy("cno").orderBy(F.col("data_da_situacao").desc())
antes_dedup = df.count()
df = (
    df.withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "municipio_valido")
)
print(f"Válidos: {antes_dedup:,} | Após dedup por cno: {df.count():,}")

df = df.select(*COLUNAS_FINAIS)
display(df.limit(10))

In [0]:
save_table(df, TARGET_TABLE)
add_column_comments(
    spark,
    TARGET_TABLE,
    SILVER_CNO_COMMENTS
)
print(f"Tabela {TARGET_TABLE} persistida: {spark.table(TARGET_TABLE).count():,} linhas")

In [0]:
total = spark.table(TARGET_TABLE).count()
distintos = spark.table(TARGET_TABLE).select("cno").distinct().count()
print(f"Total: {total:,} | CNO distintos: {distintos:,} | Duplicatas: {total - distintos:,}")
assert total == distintos, "Quebra de unicidade de cno em silver.cno"
display(spark.sql(f"SELECT situacao, count(*) AS qtd_obras FROM {TARGET_TABLE} GROUP BY situacao ORDER BY situacao"))
display(spark.sql(f"SELECT min(data_de_inicio) AS primeira_inicio, max(data_de_inicio) AS ultima_inicio FROM {TARGET_TABLE}"))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))